In [1]:
#Import necessary modules
import pandas as pd
import numpy as np
import ast
from scipy.stats import ranksums,wilcoxon
#kmeans
from sklearn.cluster import KMeans
from sklearn import preprocessing
from sklearn.metrics import silhouette_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
import statistics
from collections import Counter

import math
from scipy.stats import variation
from scipy.stats import iqr
from scipy import stats 

import random
from random import seed
from random import randint
from sklearn.neighbors import LocalOutlierFactor
from pyod.models.lof import LOF
from pyod.models.ocsvm import OCSVM
from sklearn.metrics import confusion_matrix
import scipy.stats
import scipy.stats as st
from scipy.stats import t

import pandas as pd
from sklearn.ensemble import IsolationForest
from statistics import median
# from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
patients=[1135, 1450, 1464, 1497, 1504, 
          1511, 1541, 1559, 1572, 1582, 
          1586, 1603, 1607, 1609, 1615, 
          1617, 1628, 1657, 1660, 1706]

In [3]:
def readData(patient):
    tm = pd.read_csv(f"{patient}_cleaned_Step2_entropy_alldata"'.csv')
    out = np.array(tm).tolist()  
    return out

In [4]:
def possible_visitor_non_adjacents(patient):
    try:
        tm = pd.read_csv(f"{patient}_possible_visitor_non_adjacent_twice_alldata"'.csv')
        possible_visitor_non_adjacent = np.array(tm).tolist()  
    except:
        possible_visitor_non_adjacent =[]
    return possible_visitor_non_adjacent

In [5]:
def mapping_location(num):
    import csv
    df = pd.read_csv(f'Patient{num}_mapping_entropy.csv')
#     convert dataframe to dictionary
#     my_dict = tm.to_dict('records')

    # Convert DataFrame to dictionary
    result_dict = df.to_dict('list')

   # Convert values to list of lists
    for key in result_dict:
        result_dict[key] = [ast.literal_eval(value) for value in result_dict[key]]
    return result_dict


### Entryway sensor firings

In [6]:
exit_rooms = ['Front Door','Back Door','Other Door',
              'Balcony Door','Other Door 2','Garage Door','Balcony 1']#'Bathroom'

### split data into train and test

In [7]:
# find date in data

def date_data(out):
    # number of sensors in each house
    all_date=[]
    for i in range(len(out)):
        dt = out[i][0].split(' ')[0]
        all_date.append(dt)

    #remove duplicate
    new_date=list(dict.fromkeys(all_date))
    return new_date

In [8]:
# split data into train and test---60days
def split_tain_test(n_init):  
    import datetime
    traning_data = []
    test_data = []
    for item in out:
        start_date =datetime.datetime.strptime(out[0][0].split(' ')[0], "%d/%m/%Y").date()   
        dt =datetime.datetime.strptime(item[0].split(' ')[0], "%d/%m/%Y").date()
        
        #last traning date
        last=new_date[n_init]
        
        last_dt =datetime.datetime.strptime(last, "%d/%m/%Y").date()
        if dt < last_dt:
#             print(start_date,dt)
            traning_data.append(item)
        else:
            test_data.append(item)
            
    return traning_data, test_data


# Permutation Entropy

In [9]:
from  pyentrp import entropy as ent
import numpy as np
import math

In [10]:
# A = [4,7,9,10,6,11,3]
# m=3
# r=1
# ent.permutation_entropy(A, order=m,delay=r,normalize=True)

In [11]:
# B=[4,7,9]
# C=[10,6,11,3]
# ent.permutation_entropy(B, order=m,delay=r,normalize=True)

In [12]:
# a,b= ent.multiscale_permutation_entropy(A, m, delay=r, scale=2)
# (a+b)/2

In [13]:
# import neurokit2 as nk
def threshold(entropy,m,r): #multi or perm
    MPE_all={}
    for date in train_date :   
        MPE_all[date]={}# training data to learn threshold
        data = ai_vectors_by_day[date]
#         PE=[]
#         MPE=[]
#         Fuzzy=[]
        means=[]
        for hour in range(24):
            MPE_all[date][hour]=[]

            pe = ent.permutation_entropy(data[hour], order=m,delay=r,normalize=True)
#             pe=nk.entropy_permutation(data[hour], delay=r, dimension=m)[0]
#             PE.append(pe)

#             a,b= ent.multiscale_permutation_entropy(data[hour], m, delay=r, scale=2)
#             mpe=(a+b)/2
# #             MPE.append(mpe)
            
#             fuzzy=nk.entropy_fuzzy(data[hour], delay=r, dimension=m)[0]
# #             Fuzzy.append(fuzzy)
            
            
            if entropy=='multi':
                MPE_all[date][hour].append(mpe)#pe
#                 print(mpe)
            elif entropy=='perm':
                if math.isnan(pe):
                    pe=0
                MPE_all[date][hour].append(pe)
#           A].append(fuzzy)
                
#                 print(pe)
        combined_dict = {}

        # Iterate over the original dictionary
        for date, hour_dict in MPE_all.items():  #
            for hour, values in hour_dict.items():
                # Check if hour already exists in combined_dict, if not create a new list
                if hour not in combined_dict:
                    combined_dict[hour] = []
                # Add all the values in the hour to the list in combined_dict
                combined_dict[hour] += values
#         print(combined_dict)
        standard_deviations_bound=[]            
        #not update threhold overtime (use all training)            
        for hour in range(24):   
            #standard deviation--sample
            standard_deviation=np.std(combined_dict[hour],ddof=1) #combined_dict[hour]
    #         print('standard_deviation',standard_deviation)
            # mu
            mu=np.mean(combined_dict[hour])
            means.append(mu)  
            #The threshold value is chosen based on the standard deviation, σ = 1. 
            threshold=mu+1*standard_deviation   
            standard_deviations_bound.append(threshold)   
    return standard_deviations_bound

In [14]:
# standard_deviations_bound_mpe=threshold('multi',3,1)     
# standard_deviations_bound_mpe

In [15]:
def firing_data_hrs(out):
    # Create an empty dictionary to store the firing data
    firing_data = {}

    # Loop through each data point and extract the date and hour of the firing
    for point in out:
       
        datetime = point[0]

        # Extract the date and hour from the datetime string
        date, time = datetime.split()
        hour = time.split(":")[0]

        # If this is the first firing for this date, create a new dictionary for it
        if date not in firing_data:
            firing_data[date] = [0 for h in range(24)]
            
        if point[1] in exit_rooms:
            firing_data[date][int(hour)] = 1
    return firing_data


In [16]:
for num in patients:
    patient = num
    out=readData(patient)
    possible_visitor_non_adjacent=possible_visitor_non_adjacents(patient)
    ai_vectors_by_day=mapping_location(num)

    new_date=date_data(out)
    # train_weeks = 10

    n_init=60

    split_output=split_tain_test(n_init)
    # train_data = split_tain_test[0]
    train_data = split_output[0]
    # test = split_tain_test[1]
    test_data = split_output[1]



    train_date=date_data(train_data)
    test_date=date_data(test_data)
    new_date=date_data(out)

    standard_deviations_bound_pe=threshold('perm',3,1)   


    #PE

#     with PdfPages(f"{num}_entropy_threshold_order_3_alldata"'.pdf') as pdf:
#         plt.figure(figsize=(8, 5))

    from pyentrp import entropy as ent
    import numpy as np
    MPE_all={}
    compare_all={}

    for date in new_date :                   # all data:to find those above the threshold
        MPE_all[date]={}
        compare_all[date]={}


#             fig, ax = plt.subplots()
        data = ai_vectors_by_day[date]
        PE=[]
        MPE=[]


        for hour in range(24):
            MPE_all[date][hour]=[]
            compare_all[date][hour]=[]


            pe = ent.permutation_entropy(data[hour], order=3,delay=1,normalize=True)
            PE.append(pe)

#             mpe, _ = ent.multiscale_permutation_entropy(data[hour], m=3, delay=1, scale=2)
#             a,b= ent.multiscale_permutation_entropy(data[hour], m=3, delay=1, scale=2)
#             mpe=(a+b)/2        
#             MPE.append(mpe)

            MPE_all[date][hour].append(pe)#pe


        for i in range(len(standard_deviations_bound_pe)):
            threshhold =standard_deviations_bound_pe[i]  # threshold for each hour
            hour = i
            pe = ent.permutation_entropy(data[hour], order=3,delay=1,normalize=True)
#             mpe, _ = ent.multiscale_permutation_entropy(data[hour], m=3, delay=1, scale=2)
    #             print(threshhold,pe,type(threshhold))
            #calculate the diff
            if threshhold>=pe:  #pe
                diff =0
            else:
    #             print(mpe,mu+2*standard_deviation)
                diff = pe-(threshhold)                      #pe

            compare_all[date][hour].append(diff)

#     #         # Plot the first list as a red dashed line
#     #         ax.plot(MPE, color='blue', linestyle='--',label='MPE')

#             # Plot the second list as a blue dashed line
#             ax.plot(PE, color='red', linestyle='--',label='PerEn')

#             # Plot the second list as a green solid line
#             ax.plot(standard_deviations_bound_pe, color='green', linestyle='-',label='Standard deviation boundaries')

#         #       # Plot the second list as a orange solid line
#         #     ax.plot(means, color='orange', linestyle='-',label='Mean of MPE')

#             # Set the x and y labels
#             ax.set_xlabel('Time (Hours)')
#             ax.set_ylabel('PerEn Values')

#             # Add a legend
#             plt.legend()
#             plt.title(f"Possible visitor events on date  {date}")
#             pdf.savefig() 

    combined_dict_diif = {}
    for date, hour_dict in compare_all.items():
        values = []
        for hour_values in hour_dict.values():
            values += hour_values
        combined_dict_diif[date] = values

#     with PdfPages(f"{num}_pe_entropy_visitors_order_3_alldata"'.pdf') as pdf:
#         plt.figure(figsize=(8, 5))

#         firing_data=firing_data_hrs(out)
#         for date, hour_dict in combined_dict_diif.items():
#             fig, ax = plt.subplots()
#             ax.plot(hour_dict)

#             for i, val in enumerate(firing_data[date]):
#                 if val == 1:
#                 # Add vertical line 
#                     ax.axvline(i, color='r')
#                     plt.title(f"Possible visitor events on {date}")
#                     plt.ylim(bottom=0) 
#                     # Set the x and y labels
#                     ax.set_xlabel('Time (Hours)')
#                     ax.set_ylabel('PerEn Entropy Value')
#             pdf.savefig() 
            
#     #save entropy to csv
    for k, v in combined_dict_diif.items():
        combined_dict_diif[k] = [np.nan if x == 0 else x for x in v]
    df_entropy=pd.DataFrame.from_dict(combined_dict_diif, orient='index').T
    df_entropy.to_csv(f"{patient}_entropy_alldata"'.csv', index=False, header=True)

/Users/yge18/opt/anaconda3/lib/python3.8/site-packages/numpy/core/_methods.py:262: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/yge18/opt/anaconda3/lib/python3.8/site-packages/numpy/core/_methods.py:254: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


In [17]:
xx

NameError: name 'xx' is not defined